# 01 — Exploratory Data Analysis
## Dynamic Pricing & Demand Forecasting
### Goal: Understand demand patterns, seasonality, and price signals in M5 retail data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sales = pd.read_csv('../data/raw/sales_train_validation.csv')
calendar = pd.read_csv('../data/raw/calendar.csv')
prices = pd.read_csv('../data/raw/sell_prices.csv')

print("Sales shape:", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)

## Section 2: Data Overview

In [ ]:
# ── Section 2: Data Overview ──────────────────────────────

# 2.1 Shape & basic info
print("=" * 50)
print("SALES FILE")
print("=" * 50)
print(f"Rows:           {sales.shape[0]:,}")
print(f"Columns:        {sales.shape[1]:,}")
print(f"Categories:     {sales['cat_id'].unique().tolist()}")
print(f"States:         {sales['state_id'].unique().tolist()}")
print(f"Stores:         {sales['store_id'].nunique()}")
print(f"Unique items:   {sales['item_id'].nunique():,}")
print(f"Null values:    {sales.isnull().sum().sum()}")

print("\n" + "=" * 50)
print("CALENDAR FILE")
print("=" * 50)
print(f"Date range:     {calendar['date'].min()} to {calendar['date'].max()}")
print(f"Total days:     {len(calendar):,}")
print(f"Event types:    {calendar['event_type_1'].dropna().unique().tolist()}")
print(f"SNAP columns:   {[c for c in calendar.columns if 'snap' in c.lower()]}")

print("\n" + "=" * 50)
print("PRICES FILE")
print("=" * 50)
print(f"Rows:           {prices.shape[0]:,}")
print(f"Price range:    ${prices['sell_price'].min():.2f} to ${prices['sell_price'].max():.2f}")
print(f"Avg price:      ${prices['sell_price'].mean():.2f}")
print(f"Null prices:    {prices['sell_price'].isnull().sum()}")

# 2.2 Items per store check
print("\n" + "=" * 50)
print("BALANCE CHECK")
print("=" * 50)
items_per_store = sales.groupby('store_id')['item_id'].nunique()
print(items_per_store)
expected = sales['item_id'].nunique() * sales['store_id'].nunique()
print(f"\nExpected rows:  {expected:,}")
print(f"Actual rows:    {len(sales):,}")
print(f"Balanced:       {expected == len(sales)}")

# 2.3 Event analysis
print("\n" + "=" * 50)
print("EVENT ANALYSIS")
print("=" * 50)
print(f"Total days:     {len(calendar):,}")
print(f"Event days:     {calendar['event_name_1'].notna().sum():,}")
print(f"Normal days:    {calendar['event_name_1'].isna().sum():,}")
print(f"\nEvents by type:")
print(calendar['event_type_1'].value_counts())

two_events = calendar[calendar['event_name_2'].notna()]
print(f"\nDays with 2 events: {len(two_events)}")
print(two_events[['date', 'event_name_1', 'event_name_2']])

### Key findings — Section 2: Data overview

- 30,490 time series = 3,049 unique items × 10 stores (balanced panel)
- 5.5 years of daily sales (Jan 2011 — Jun 2016)
- 3 categories: FOODS, HOUSEHOLD, HOBBIES
- 3 states: California, Texas, Wisconsin
- Price range $0.01–$107.32 (avg $4.41) — log transform needed later
- Zero nulls across all 3 files — clean dataset
- 162 event days (8.2%) — Religious (55) > National (52) > Cultural (37) > Sporting (18)
- Only 5 days had 2 simultaneous events — event_name_2 safely ignored

## Section 3: Demand Patterns over Time
Goal: Spot seasonality, trends, and anomalies across 1,913 days

In [ ]:
sales.head()

In [ ]:
# Melt from wide to long format
sales_long = sales.melt(
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    value_vars=[c for c in sales.columns if c.startswith('d_')],
    var_name='day_id',
    value_name='sales'
)

print("Long format shape:", sales_long.shape)
print(sales_long.head(3))

In [ ]:
calendar.head()

In [ ]:
sales_long = sales_long.merge(
    calendar[['d', 'date', 'wm_yr_wk', 'weekday', 
              'month', 'year', 'event_name_1', 
              'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']],
    left_on='day_id',
    right_on='d',
    how='left'
)

sales_long.drop(columns=['d'], inplace=True)
sales_long['date'] = pd.to_datetime(sales_long['date'])

print("Shape after calendar merge:", sales_long.shape)
print(sales_long[['item_id', 'store_id', 'date', 'sales', 'snap_CA']].head(3))

In [ ]:
# Aggregate total sales per day across all items and stores
daily_sales = sales_long.groupby('date')['sales'].sum().reset_index()
daily_sales.columns = ['date', 'total_sales']

# Plot
fig = px.line(
    daily_sales,
    x='date',
    y='total_sales',
    title='Total daily demand — all items, all stores (2011–2016)',
    labels={'total_sales': 'Total units sold', 'date': 'Date'}
)
fig.update_traces(line_width=0.8, line_color='#378ADD')
fig.update_layout(height=400)
fig.show()

print(f"\nAvg daily sales: {daily_sales['total_sales'].mean():,.0f} units")
print(f"Max daily sales: {daily_sales['total_sales'].max():,.0f} units on {daily_sales.loc[daily_sales['total_sales'].idxmax(), 'date'].date()}")
print(f"Min daily sales: {daily_sales['total_sales'].min():,.0f} units on {daily_sales.loc[daily_sales['total_sales'].idxmin(), 'date'].date()}")

In [ ]:
# Weekly aggregation — smoother signal
weekly_sales = sales_long.groupby(
    pd.Grouper(key='date', freq='W')
)['sales'].sum().reset_index()

fig = px.line(
    weekly_sales,
    x='date',
    y='sales',
    title='Weekly total demand — all items, all stores (2011–2016)',
    labels={'sales': 'Total units sold', 'date': 'Date'}
)
fig.update_traces(line_width=1.2, line_color='#1D9E75')
fig.update_layout(height=400)
fig.show()

In [ ]:
yearly = sales_long.groupby('year')['sales'].sum().reset_index()
yearly.columns = ['year', 'total_sales']

fig = px.bar(
    yearly,
    x='year',
    y='total_sales',
    title='Total annual sales by year',
    labels={'total_sales': 'Total units sold', 'year': 'Year'},
    color='total_sales',
    color_continuous_scale='Blues'
)
fig.update_layout(height=350, showlegend=False)
fig.show()

print(yearly)

### Key findings: Demand patterns over time
- Avg daily demand: 34,342 units across all stores and items
  (34,342 ÷ 30,490 series = ~1.13 units per item-store per day)
- Full year growth (2012→2015): 12M → 13.8M units = +15%
- 2013 vs 2014 nearly flat (+0.3%) — no clear explanation yet
- 2011 and 2016 are partial years — excluded from year-over-year comparisons
- Sharp daily drops coincide with major holidays (min: 11 units on 2012-12-25)
- Per-item demand is highly intermittent (~1.13 units/day) — key modeling challenge

## Section 4: Category & Store Analysis
Goal: Understand which categories and stores drive the most demand

In [ ]:
# Total sales by category
cat_sales = sales_long.groupby('cat_id')['sales'].sum().reset_index()
cat_sales.columns = ['category', 'total_sales']
cat_sales['pct'] = (cat_sales['total_sales'] / cat_sales['total_sales'].sum() * 100).round(1)
cat_sales = cat_sales.sort_values('total_sales', ascending=False)

fig = px.bar(
    cat_sales,
    x='category',
    y='total_sales',
    title='Total sales by category (2011–2016)',
    labels={'total_sales': 'Total units sold', 'category': 'Category'},
    color='category',
    color_discrete_map={
        'FOODS': '#378ADD',
        'HOUSEHOLD': '#1D9E75',
        'HOBBIES': '#BA7517'
    },
    text='pct'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=600, showlegend=False)
fig.show()

print(cat_sales)

In [ ]:
# Total sales by state
state_sales = sales_long.groupby('state_id')['sales'].sum().reset_index()
state_sales.columns = ['state', 'total_sales']
state_sales['pct'] = (state_sales['total_sales'] / state_sales['total_sales'].sum() * 100).round(1)
state_sales = state_sales.sort_values('total_sales', ascending=False)

fig = px.bar(
    state_sales,
    x='state',
    y='total_sales',
    title='Total sales by state (2011–2016)',
    labels={'total_sales': 'Total units sold', 'state': 'State'},
    color='state',
    color_discrete_map={
        'CA': '#378ADD',
        'TX': '#1D9E75',
        'WI': '#BA7517'
    },
    text='pct'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=600, showlegend=False)
fig.show()

print(state_sales)

In [ ]:
# Sales by individual store
store_sales = sales_long.groupby('store_id')['sales'].sum().reset_index()
store_sales.columns = ['store', 'total_sales']
store_sales = store_sales.sort_values('total_sales', ascending=False)

fig = px.bar(
    store_sales,
    x='store',
    y='total_sales',
    title='Total sales by store (2011–2016)',
    labels={'total_sales': 'Total units sold', 'store': 'Store'},
    color='total_sales',
    color_continuous_scale='Blues'
)
fig.update_layout(height=400)
fig.show()

print(store_sales)

In [ ]:
# How each category trends over time
cat_daily = sales_long.groupby(
    ['date', 'cat_id']
)['sales'].sum().reset_index()

fig = px.line(
    cat_daily,
    x='date',
    y='sales',
    color='cat_id',
    title='Daily demand by category (2011–2016)',
    labels={'sales': 'Total units sold', 'date': 'Date', 'cat_id': 'Category'},
    color_discrete_map={
        'FOODS': '#378ADD',
        'HOUSEHOLD': '#1D9E75',
        'HOBBIES': '#BA7517'
    }
)
fig.update_traces(line_width=0.8)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Top 10 items by total sales
top_items = sales_long.groupby(
    ['item_id', 'cat_id']
)['sales'].sum().reset_index()
top_items.columns = ['item_id', 'category', 'total_sales']
top_items = top_items.sort_values('total_sales', ascending=False).head(10)

fig = px.bar(
    top_items,
    x='total_sales',
    y='item_id',
    orientation='h',
    title='Top 10 best selling items (all stores, 2011–2016)',
    labels={'total_sales': 'Total units sold', 'item_id': 'Item'},
    color='category',
    color_discrete_map={
        'FOODS': '#378ADD',
        'HOUSEHOLD': '#1D9E75',
        'HOBBIES': '#BA7517'
    }
)
fig.update_layout(height=400)
fig.show()

print(top_items)

### Key findings : Category & store analysis

- FOODS dominates at 68.6% — primary modeling focus
- HOUSEHOLD 22%, HOBBIES 9.3% — flattest growth, most intermittent
- CA leads at 43.6% — 4 stores AND higher per-store volume (10.9% vs 9.6% TX)
- TX (28.8%) and WI (27.6%) nearly equal despite being different states
- CA_3 is highest volume store at 11.2M units — 45% more than CA_1
- CA_4 is lowest volume store — large within-state variance in CA
- All top 10 items are FOODS_3 — no HOUSEHOLD or HOBBIES in top 10
- FOODS_3_090 is #1 item at 1M+ units across all stores 2011-2016
- FOODS likely dominates WRMSSE weighting due to volume — to be confirmed in evaluation notebook

## Section 5: Calendar & Event Effects
Goal: Quantify how holidays, events, and SNAP days impact demand

In [ ]:
sales_long['weekday'].unique()

In [ ]:
# Average sales by day of week
dow_sales = sales_long.groupby('weekday')['sales'].mean().reset_index()
dow_sales.columns = ['weekday', 'avg_sales']

# Order days correctly
day_order = ['Monday', 'Tuesday', 'Wednesday', 
             'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales['weekday'] = pd.Categorical(
    dow_sales['weekday'], categories=day_order, ordered=True
)
dow_sales = dow_sales.sort_values('weekday')

fig = px.bar(
    dow_sales,
    x='weekday',
    y='avg_sales',
    title='Average demand by day of week',
    labels={'avg_sales': 'Avg units sold per item', 'weekday': 'Day'},
    color='avg_sales',
    color_continuous_scale='Blues'
)
fig.update_layout(height=350, showlegend=False)
fig.show()

print(dow_sales)

In [ ]:
# Average sales by month
month_sales = sales_long.groupby('month')['sales'].mean().reset_index()
month_sales.columns = ['month', 'avg_sales']

month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
month_sales['month_name'] = month_sales['month'].map(month_names)

fig = px.line(
    month_sales,
    x='month',
    y='avg_sales',
    title='Average demand by month (seasonality)',
    labels={'avg_sales': 'Avg units sold per item', 'month': 'Month'},
    markers=True
)
fig.update_traces(line_color='#378ADD', line_width=2)
fig.update_layout(height=350)
fig.update_xaxes(
    tickvals=list(range(1,13)),
    ticktext=list(month_names.values())
)
fig.show()

print(month_sales)

In [ ]:
# ── Event impact analysis ─────────────────────────────────
import plotly.graph_objects as go

# Create is_event flag
sales_long['is_event'] = sales_long['event_name_1'].notna().astype(int)

# Compare avg sales on event vs non-event days
event_impact = sales_long.groupby('is_event')['sales'].mean().reset_index()
event_impact['label'] = event_impact['is_event'].map({0: 'Normal day', 1: 'Event day'})
event_impact['pct_diff'] = ((event_impact['sales'] / event_impact.loc[0, 'sales'] - 1) * 100).round(1)

# ── Plot ──────────────────────────────────────────────────
colors = {'Normal day': '#378ADD', 'Event day': '#BA7517'}

fig = go.Figure()

for _, row in event_impact.iterrows():
    pct   = row['pct_diff']
    label = f"+{pct}%" if pct > 0 else f"{pct}%" if pct != 0 else "baseline"

    fig.add_trace(go.Bar(
        x=[row['label']],
        y=[row['sales']],
        marker_color=colors[row['label']],
        text=label,
        textposition='outside',
        name=row['label'],
        hovertemplate=(
            f"<b>{row['label']}</b><br>"
            f"Avg sales: {row['sales']:.3f} units<br>"
            f"vs normal: {label}<extra></extra>"
        )
    ))

fig.update_layout(
    title={
        'text': 'Event days average 5.3% lower demand than normal days<br>'
                '<sup>Note: aggregates all event types — breakdown by type in next chart</sup>',
        'x': 0.5
    },
    yaxis=dict(
        title='Avg units sold per item',
        range=[0, event_impact['sales'].max() * 1.2]
    ),
    xaxis=dict(title=''),
    showlegend=False,
    height=400,
    annotations=[
        dict(
            x=0.5, y=-0.18,
            xref='paper', yref='paper',
            text='⚠️ Misleading aggregate — national holiday closures dominate. See event type breakdown below.',
            showarrow=False,
            font=dict(size=11, color='gray'),
            align='center'
        )
    ]
)

fig.show()
print(event_impact[['label', 'sales', 'pct_diff']])

In [ ]:
# Impact by event type
event_type_sales = sales_long[sales_long['is_event']==1].groupby(
    'event_type_1'
)['sales'].mean().reset_index()
event_type_sales.columns = ['event_type', 'avg_sales']

# Add normal day baseline
baseline = sales_long[sales_long['is_event']==0]['sales'].mean()
event_type_sales = event_type_sales.sort_values('avg_sales', ascending=False)

fig = px.bar(
    event_type_sales,
    x='event_type',
    y='avg_sales',
    title='Average demand by event type',
    labels={'avg_sales': 'Avg units sold per item', 'event_type': 'Event type'},
    color='event_type',
    color_discrete_map={
        'Religious': '#534AB7',
        'National':  '#378ADD',
        'Cultural':  '#1D9E75',
        'Sporting':  '#BA7517'
    }
)
fig.add_hline(
    y=baseline,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Normal day baseline: {baseline:.3f}'
)
fig.update_layout(height=380, showlegend=False)
fig.show()

print(f"\nNormal day baseline: {baseline:.4f}")
print(event_type_sales)

In [ ]:
sales_long.head()

In [ ]:
# ── Create is_snap feature ──────────────────────────
sales_long['is_snap'] = np.select(
    condlist=[
        sales_long['state_id'] == 'CA',
        sales_long['state_id'] == 'TX',
        sales_long['state_id'] == 'WI'
    ],
    choicelist=[
        sales_long['snap_CA'],
        sales_long['snap_TX'],
        sales_long['snap_WI']
    ]
)

# ── Overall SNAP lift ─────────────────────────────────────────────
snap_impact = sales_long.groupby('is_snap')['sales'].mean().reset_index()
snap_impact['label'] = snap_impact['is_snap'].map({0: 'Non-SNAP day', 1: 'SNAP day'})
snap_lift = ((snap_impact.loc[1, 'sales'] / snap_impact.loc[0, 'sales'] - 1) * 100).round(1)

snap_impact['text'] = snap_impact['is_snap'].map({
    0: 'baseline',
    1: f'+{snap_lift}%'
})

fig = px.bar(
    snap_impact,
    x='label',
    y='sales',
    title=f'SNAP days drive +{snap_lift}% higher demand — concentrated in FOODS<br>'
          f'<sup>Non-SNAP avg: {snap_impact.loc[0,"sales"]:.3f} → SNAP avg: {snap_impact.loc[1,"sales"]:.3f} units per item</sup>',
    labels={'sales': 'Avg units sold per item', 'label': ''},
    color='label',
    color_discrete_map={
        'Non-SNAP day': '#378ADD',
        'SNAP day':     '#1D9E75'
    },
    text='text'
)
fig.update_traces(
    textposition='outside',
    textfont=dict(size=16, color='black')
)
fig.update_layout(
    height=450,
    showlegend=False,
    yaxis=dict(range=[0, snap_impact['sales'].max() * 1.3])
)
fig.show()

# ── SNAP impact by category ───────────────────────────────────────
snap_cat = sales_long.groupby(['is_snap', 'cat_id'])['sales'].mean().reset_index()
snap_cat['label'] = snap_cat['is_snap'].map({0: 'Non-SNAP', 1: 'SNAP'})

# per-category lift
cat_lifts = {}
for cat in snap_cat['cat_id'].unique():
    non_snap = snap_cat[(snap_cat['cat_id'] == cat) & (snap_cat['is_snap'] == 0)]['sales'].values[0]
    snap_val = snap_cat[(snap_cat['cat_id'] == cat) & (snap_cat['is_snap'] == 1)]['sales'].values[0]
    cat_lifts[cat] = round((snap_val / non_snap - 1) * 100, 1)

fig2 = px.bar(
    snap_cat,
    x='cat_id',
    y='sales',
    color='label',
    barmode='group',
    title='SNAP day impact by category<br>'
          '<sup>Only FOODS shows meaningful lift — SNAP cannot be spent on non-food items</sup>',
    labels={'sales': 'Avg units sold', 'cat_id': 'Category'},
    color_discrete_map={
        'Non-SNAP': '#378ADD',
        'SNAP':     '#1D9E75'
    }
)

# baseline reference line
baseline = snap_cat[snap_cat['is_snap'] == 0]['sales'].mean()
fig2.add_hline(
    y=baseline,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Non-SNAP baseline: {baseline:.3f}',
    annotation_position='top right'
)

# per-category lift annotations
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    lift  = cat_lifts.get(cat, 0)
    label = f'+{lift}%' if lift > 0 else f'{lift}%'
    fig2.add_annotation(
        x=cat,
        y=snap_cat[snap_cat['cat_id'] == cat]['sales'].max() + 0.08,
        text=f'<b>{label}</b>',
        showarrow=False,
        font=dict(size=14, color='black')
    )

fig2.update_layout(height=500)
fig2.show()

print(f"Overall SNAP lift: +{snap_lift}%")
print("\nSNAP lift by category:")
for cat, lift in cat_lifts.items():
    print(f"  {cat}: +{lift}%")

## Section 5: Key Findings — Calendar & Event Effects

## Day of week
- Saturday (1.363) and Sunday (1.349) are peak demand days
- Wednesday (0.984) is lowest demand day, Tuesday (0.996) close behind
- Weekend premium: +38% vs Wednesday low — strong weekly seasonality
- Friday (1.123) shows transition effect — people shopping after work
- day_of_week will be a top LightGBM feature

## Monthly seasonality
- August is peak demand month (1.179 avg units/item/day) — back to school season
- May is lowest demand month (1.066)
- Clear summer surge June–August
- December second lowest (1.082) — Christmas Day closures pull monthly average down
- Monthly variation (~10%) weaker than weekly variation (~38%)
- month feature captures non-linear seasonality pattern

## Event effects
- Overall event days: -5.3% vs normal — misleading aggregate
- National events biggest drop (-14.6%) — store closures on Independence Day, Thanksgiving
- Sporting events only type above baseline (+3.8%) — SuperBowl snack buying effect
- Cultural (-0.7%) and Religious (-2.1%) essentially neutral
- Four separate flags to be engineered in notebook 02: 
  is_sporting, is_national, is_religious, is_cultural
- One flag per event type preserves the opposite effects 
  (sporting = positive, national = negative)

## SNAP effects
- Overall SNAP lift: +12.7%
- FOODS: +17.2% — direct SNAP spending (drives almost all of overall lift)
- HOBBIES: +2.4%, HOUSEHOLD: +3.5% — small foot traffic spillover only
- SNAP benefits can only be spent on food — non-food lift is incidental
- Three state SNAP columns collapsed into single is_snap per row using state_id
- is_snap × FOODS interaction will be a strong feature — to add in notebook 02

In [ ]:
## Section 6: Price Analysis
# Chart 1 — price distribution by category

# First merge prices with category info from sales_long
price_cat = prices.merge(
    sales_long[['item_id', 'store_id', 'cat_id']].drop_duplicates(),
    on=['item_id', 'store_id'],
    how='left'
)

fig = px.histogram(
    price_cat,
    x='sell_price',
    color='cat_id',
    nbins=100,
    title='Price distribution by category',
    labels={'sell_price': 'Price ($)', 'cat_id': 'Category'},
    color_discrete_map={
        'FOODS': '#378ADD',
        'HOUSEHOLD': '#1D9E75',
        'HOBBIES': '#BA7517'
    },
    barmode='overlay',
    opacity=0.7
)
fig.update_layout(height=400)
fig.show()

print("\nAvg price by category:")
print(price_cat.groupby('cat_id')['sell_price'].describe().round(2))

In [ ]:
# Chart 2 — price vs demand scatter
# Sample 50k rows for speed
sample = sales_long[sales_long['sales'] > 0].merge(
    prices,
    on=['item_id', 'store_id', 'wm_yr_wk'],
    how='left'
).sample(50000, random_state=42)

fig = px.scatter(
    sample,
    x='sell_price',
    y='sales',
    color='cat_id',
    title='Price vs demand (sample 50k rows, non-zero sales only)',
    labels={'sell_price': 'Price ($)', 'sales': 'Units sold'},
    color_discrete_map={
        'FOODS': '#378ADD',
        'HOUSEHOLD': '#1D9E75',
        'HOBBIES': '#BA7517'
    },
    opacity=0.3
)
fig.update_layout(height=400)
fig.show()

# Correlation
corr = sample[['sell_price', 'sales']].corr().iloc[0,1]
print(f"\nPrice vs demand correlation: {corr:.3f}")

## Key findings: Price analysis

- FOODS avg price $3.25 — cheapest category, low variance (std $2.13), max $19.48

- HOBBIES avg price $5.33 — mid range, highest variance (std $4.83), max $30.98

- HOUSEHOLD avg price $5.47 — mid range, widest max at $107.32 (likely large appliance)

- All categories right-skewed — most items cluster under $10, long tail to the right

- Log transform not needed for LightGBM (tree-based, scale invariant) but needed for elasticity model in notebook 08

- Price vs demand raw correlation: -0.181 — negative but weak

- Weak because confounders dominate: day of week, seasonality, store, SNAP, events

- True price elasticity requires controlling for confounders — planned for notebook 08

- $0.01 minimum across all categories — potential data quality issue

## Section 7: EDA Summary — Key Findings for Modeling

### Data overview
- 30,490 time series = 3,049 unique items × 10 stores (balanced panel)
- 58.3M rows after melting to long format
- 5.5 years daily sales (Jan 2011 — Apr 2016)
- Zero nulls across all three raw files — clean dataset
- 2011 and 2016 are partial years — excluded from year-over-year comparisons

### Demand patterns
- Average daily demand: 34,342 units across all stores
- 15% growth from 2012 to 2015 (12M to 13.8M annual units)
- Per-item demand: ~1.13 units/day — highly intermittent, especially HOBBIES
- Sharp daily drops on major holidays — minimum 11 units on 2012-12-25

### Category and store
- FOODS dominates at 68.6% of total sales
- HOUSEHOLD 22%, HOBBIES 9.3% — flattest growth, most intermittent
- CA largest state (43.6%) — 4 stores AND higher per-store volume than TX/WI
- CA_3 highest volume store (11.2M units, 45% above CA_1), CA_4 lowest
- All top 10 best selling items from FOODS_3 department
- Model accuracy on FOODS disproportionately drives overall WRMSSE score

### Calendar and events
- Saturday (+38% vs Wednesday low) and Sunday are peak demand days
- Friday transition effect (+11% vs midweek) — post-work shopping
- August peak month (1.179), May lowest (1.066) — summer surge confirmed
- Monthly variation (~10%) weaker than weekly variation (~38%)
- National events: -14.6% — strongest event signal, store closures
- Sporting events: +3.8% — only event type above baseline
- Religious: -2.1%, Cultural: -0.7% — weak but included
- Four separate flags to engineer in notebook 02: is_national, is_sporting, is_religious, is_cultural
- Improvement identified: days_before_holiday to capture pre-holiday demand spike

### SNAP effects
- Overall SNAP lift: +12.7%
- FOODS specifically: +17.2% — direct SNAP spending
- HOBBIES +2.4%, HOUSEHOLD +3.5% — small foot traffic spillover only
- Three state SNAP columns collapsed into single is_snap per row using state_id
- is_snap x FOODS interaction expected to be one of strongest features

### Price analysis
- FOODS avg $3.25 — cheapest, low variance (std $2.13), max $19.48
- HOBBIES avg $5.33 — highest variance (std $4.83), max $30.98
- HOUSEHOLD avg $5.47 — widest max at $107.32
- All categories right-skewed — log transform NOT needed for LightGBM (tree-based)
  but will be needed for elasticity model in notebook 08
- Price vs demand raw correlation: -0.181 — negative but weak
- Weak because confounders dominate — true elasticity planned for notebook 08

### Why LightGBM?
- Scale: 30,490 series makes per-series classical models impractical
- Rich exogenous signals: SNAP, events, day of week, price — classical models cannot use them
- Intermittent demand: sparse count data suits tweedie objective
- Cross-series learning: learns generalizable rules across all series simultaneously
- Confirmed by M5 competition: all top 50 finishers used LightGBM
  Classical models tried and abandoned early on this exact dataset

### Top features for LightGBM (predicted)
1. lag_7, lag_28 — item's own recent history
2. rmean_7, rmean_28 — rolling demand baseline
3. rmean_28_store, rmean_28_cat — aggregated demand signal
4. day_of_week — +38% weekend premium
5. is_snap_foods — +17.2% FOODS SNAP lift
6. sell_price — price signal
7. month — summer surge, May trough
8. is_national — strongest event signal (-14.6%)
9. is_sporting — only positive event type (+3.8%)
10. days_before_holiday — pre-holiday demand spike
11. cat_id, store_id, state_id — identity features
12. price_change_pct — price momentum signal